# 01 — Embeddings exploration

**Purpose:** understand the embedding model before committing to it.

- Embed a few sample resumes and job descriptions
- Inspect vector dimensionality and cosine similarity between obvious pairs
- Sanity-check that similar roles land close together and unrelated ones do not

Findings here justify the embedding model choice in `src/config.py`
(`sentence-transformers/all-MiniLM-L6-v2` — local, free, 384-dim).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from src.search.embed import get_embeddings, embed_texts

embeddings = get_embeddings()
print('Embedding model loaded.')

In [ ]:
# A handful of resume-shaped snippets and job-shaped snippets to sanity-check the model.
texts = {
    'resume_data_analyst': 'SQL, Python, Pandas, Excel, Power BI, Statistics. Data analytics internship building dashboards.',
    'resume_ml_engineer': 'Python, PyTorch, deep learning, computer vision, model deployment, Docker, Kubernetes.',
    'resume_frontend': 'JavaScript, React, CSS, HTML, responsive design, REST API integration.',
    'job_data_analyst': 'Title: Data Analyst. Skills: SQL, Excel, Power BI, Statistics. Analyze sales data and build dashboards.',
    'job_ml_engineer': 'Title: Machine Learning Engineer. Skills: Python, PyTorch, FAISS, LangChain. Build and deploy ML models.',
    'job_frontend': 'Title: Frontend Developer. Skills: JavaScript, React, CSS, HTML. Build responsive web interfaces.',
    'job_unrelated': 'Title: Warehouse Forklift Operator. Skills: Forklift certification, physical stamina. Move pallets in a warehouse.',
}

vectors = {k: np.array(embed_texts([v])[0]) for k, v in texts.items()}
print('Vector dimensionality:', len(next(iter(vectors.values()))))

In [ ]:
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

pairs = [
    ('resume_data_analyst', 'job_data_analyst'),   # expect: high
    ('resume_ml_engineer', 'job_ml_engineer'),      # expect: high
    ('resume_frontend', 'job_frontend'),            # expect: high
    ('resume_data_analyst', 'job_ml_engineer'),     # expect: medium (some skill overlap: SQL/Python)
    ('resume_frontend', 'job_data_analyst'),        # expect: low
    ('resume_ml_engineer', 'job_unrelated'),        # expect: lowest
]

print(f"{'pair':45s}  cosine similarity")
for a, b in pairs:
    sim = cosine_sim(vectors[a], vectors[b])
    print(f'{a} <-> {b}'.ljust(45), f'{sim:.4f}')

### Expected pattern

Matching resume/job pairs (analyst-analyst, ML-ML, frontend-frontend) should score
noticeably higher than cross-domain pairs, and the unrelated warehouse job should
score lowest of all. If this pattern doesn't hold, that's a signal to try a larger
embedding model (e.g. `all-mpnet-base-v2`) — see `02b_embedding_model_comparison.ipynb`
for a head-to-head comparison against the actual job corpus.

In [ ]:
# Chunk-size sanity check on a real career note, using the same splitter the
# pipeline uses (src/parsing/loader.py).
from src.parsing.loader import load_file, chunk_documents

sample_note = Path.cwd().parent / 'data' / 'career_notes' / 'switching_to_data_analyst.txt'
text = load_file(sample_note)
print(f'Document length: {len(text)} chars')

for size, overlap in [(500, 100), (1000, 200), (1500, 300)]:
    chunks = chunk_documents([{'text': text, 'metadata': {}}], chunk_size=size, chunk_overlap=overlap)
    print(f'chunk_size={size}, overlap={overlap} -> {len(chunks)} chunk(s)')

### Conclusion

`chunk_size=1000, chunk_overlap=200` (the defaults in `.env.example` / `src/config.py`)
keeps most career notes to 1-2 chunks — enough context per chunk for the mentor to
answer from a single retrieved passage, without diluting the embedding with unrelated
content. This is the setting used in `build_index()` throughout the project.